# Conciliação de faturamento: o que foi faturado e não aparece na devolutiva do cliente

**Pergunta de negócio:** das viagens faturadas na quinzena, para os veículos da minha lista e com destino ao cliente X, quais **não aparecem** na planilha de devolutiva (ou seja, não foram reconhecidas/pagas)?

- **Planilha B (faturamento):** tudo o que foi faturado. Serve para **filtrar**.
- **Planilha A (devolutiva do cliente):** o que o cliente reconheceu. Serve para **verificar**.
- **Resultado:** linhas da B, dentro do período, cuja nota fiscal **não** está na A.

> Os dados desta pasta (`exemplos/`) são **100% fictícios**. Para usar com dados reais, altere apenas o bloco de parâmetros.

## 1. Parâmetros (só este bloco muda a cada quinzena)

In [ ]:
import pandas as pd
from pathlib import Path

# --- Arquivos de entrada ---
ARQ_A = "exemplos/devolutiva_exemplo.xlsx"    # devolutiva do cliente (Excel)
ARQ_B = "exemplos/faturamento_exemplo.csv"    # faturamento geral (CSV)
SEP_B = ";"                                   # separador do CSV
ENCODING_B = "utf-8"                          # com dados reais costuma ser "latin-1"

# --- Lista de placas ---
# Dados reais ficam em placas.txt (ignorado pelo Git). Sem ele, usa o exemplo.
ARQ_PLACAS = "placas.txt" if Path("placas.txt").exists() else "exemplos/placas_exemplo.txt"

# --- Período da quinzena ---
INICIO = pd.Timestamp("2026-09-16")
FIM = pd.Timestamp("2026-09-30")

# --- Cliente (busca por trecho, sem diferenciar maiúsculas/acentos comuns) ---
CLIENTE_REGEX = "CLIENTE X"

# --- Nomes das colunas ---
COL_NOTA_A = "Nota Fiscal"
COL_NOTA_B = "N.º - Nota Fiscal"
COL_DATA_B = "Emissão NF"
COL_DEST_B = "Razão Social - Destinatário"
PREFIXO_PLACA_B = "Placa"   # todas as colunas que começam assim entram no filtro

SAIDA = "saida/resultado_conciliacao.xlsx"

## 2. Leitura dos dados

In [ ]:
df_a = pd.read_excel(ARQ_A)
df_b = pd.read_csv(ARQ_B, sep=SEP_B, encoding=ENCODING_B)

print("A (devolutiva):  ", df_a.shape)
print("B (faturamento): ", df_b.shape)

## 3. Funções de limpeza

Dois problemas reais que o código resolve:

1. **Placas** escritas de formas diferentes (`ABC-1234`, `abc1234`, `ABC 1234`) → padronizamos: maiúsculas, sem hífen e sem espaço.
2. **Notas fiscais** com: várias notas na mesma célula, zeros à esquerda (`001234`), sufixo após hífen (`1234-1`) e mistura de número e texto na mesma coluna → separamos, limpamos e comparamos pelo número base.

In [ ]:
def normaliza_placa(serie):
    """Maiúsculas, sem espaços e sem hífens."""
    return serie.astype("string").str.upper().str.replace(r"[\s\-]", "", regex=True)


def explode_notas(df, coluna):
    """Uma nota por linha: separa células com várias notas e
    remove espaços, zeros à esquerda e o sufixo após hífen."""
    t = df[[coluna]].copy()
    t["nf"] = t[coluna].astype(str).str.split(r"[,;\s]+", regex=True)
    t = t.explode("nf")
    t["nf"] = t["nf"].str.strip().str.lstrip("0")
    t["nf"] = t["nf"].str.split("-").str[0]
    return t[t["nf"] != ""]

## 4. Filtros na Planilha B: placas da lista **e** cliente

Uma linha é mantida se **qualquer** coluna de placa (cavalo ou carreta) estiver na lista.

In [ ]:
placas = set(normaliza_placa(pd.Series(Path(ARQ_PLACAS).read_text().split())))
print(len(placas), "placas na lista")

cols_placa = [c for c in df_b.columns if c.startswith(PREFIXO_PLACA_B)]
print("Colunas de placa usadas:", cols_placa)

mask_placa = pd.Series(False, index=df_b.index)
for c in cols_placa:
    mask_placa = mask_placa | normaliza_placa(df_b[c]).isin(placas)

mask_cliente = df_b[COL_DEST_B].str.contains(CLIENTE_REGEX, case=False, na=False)

b_filtrada = df_b[mask_placa & mask_cliente].copy()

print("Só cliente:      ", mask_cliente.sum())
print("Só placas:       ", mask_placa.sum())
print("Cliente + placas:", len(b_filtrada))

## 5. Notas pagas (A) × notas da B

Usamos `isin` (e não `merge`) porque a pergunta é só "existe ou não?". Um `merge` multiplicaria linhas quando a mesma nota aparece várias vezes, inflando valores.

In [ ]:
notas_pagas = set(explode_notas(df_a, COL_NOTA_A)["nf"])
print(len(notas_pagas), "notas distintas na devolutiva")

b_nf = explode_notas(b_filtrada, COL_NOTA_B)
b_nf["paga"] = b_nf["nf"].isin(notas_pagas)

resumo = b_nf.groupby(level=0)["paga"].agg(["sum", "count"])
resumo["status"] = "parcial"                                   # só algumas notas da linha
resumo.loc[resumo["sum"] == 0, "status"] = "nao_paga"          # nenhuma nota na A
resumo.loc[resumo["sum"] == resumo["count"], "status"] = "paga"  # todas as notas na A

b_filtrada = b_filtrada.drop(columns="status", errors="ignore").join(resumo["status"])
b_filtrada["status"] = b_filtrada["status"].fillna("sem_nota")
b_filtrada["status"].value_counts()

## 6. Período da quinzena

Algumas células trazem várias datas (`22/09/2026;22/09/2026`). Pegamos a primeira, depois de **testar** que todas as datas de uma mesma célula são iguais.

In [ ]:
datas = b_filtrada[COL_DATA_B].astype(str).str.split(r"[;,]")
divergentes = (datas.apply(lambda x: len({i.strip() for i in x}) > 1)).sum()
print("Células com datas diferentes entre si:", divergentes, "(esperado: 0)")

b_filtrada["data"] = pd.to_datetime(datas.str[0].str.strip(), dayfirst=True, errors="coerce")

no_periodo = b_filtrada[(b_filtrada["data"] >= INICIO) & (b_filtrada["data"] <= FIM)]
fora_periodo = b_filtrada[(b_filtrada["data"] < INICIO) | (b_filtrada["data"] > FIM)]
sem_data = b_filtrada[b_filtrada["data"].isna()]

# Conferência: os três grupos não se sobrepõem e cobrem tudo
assert len(no_periodo) + len(fora_periodo) + len(sem_data) == len(b_filtrada)
print(len(no_periodo), "no período |", len(fora_periodo), "fora |", len(sem_data), "sem data")
no_periodo["status"].value_counts()

## 7. Validação independente (busca por trecho)

O `isin` depende da limpeza estar certa. Esta checagem **não depende dela**: procura o número como trecho de texto na planilha A inteira. Se a lista vier vazia, os dois caminhos concordam.

In [ ]:
texto_a = df_a[COL_NOTA_A].astype(str)
nao_pagas_nf = explode_notas(no_periodo[no_periodo["status"] == "nao_paga"], COL_NOTA_B)["nf"]

suspeitas = [n for n in nao_pagas_nf if texto_a.str.contains(n, regex=False).any()]
print("Notas 'não pagas' que aparecem na A por busca de trecho:", suspeitas)

## 8. Exportação

Abas: **nao_pagas_periodo** (nenhuma nota ou só parte delas na A) e **pagas_periodo**. Linhas fora do período são ignoradas.

In [ ]:
Path(SAIDA).parent.mkdir(exist_ok=True)

with pd.ExcelWriter(SAIDA) as writer:
    no_periodo[no_periodo["status"] != "paga"].to_excel(
        writer, sheet_name="nao_pagas_periodo", index=False)
    no_periodo[no_periodo["status"] == "paga"].to_excel(
        writer, sheet_name="pagas_periodo", index=False)

print("Arquivo gerado:", SAIDA)